# 03 - Outcome Labeling

**Purpose:** Develop and validate the outcome taxonomy for Kenyan judgments.

This is the **highest-risk component** in the pipeline. Getting the labels wrong
corrupts the entire model.

**Steps:**
1. Run the rule-based outcome parser on all extracted texts
2. Review confidence distribution
3. Manual review of low-confidence and conflicting cases
4. Inter-annotator agreement (Cohen's kappa target: >0.80)
5. Final binary label assignment

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from configs.settings import settings
from src.features.outcome_parser import OutcomeParser
from src.data.validators import OutcomeLabel

## 1. Load All Extracted Texts

In [ ]:
texts = {}
for txt_path in sorted(settings.data.interim_dir.rglob('*.txt')):
    case_id = txt_path.stem
    texts[case_id] = txt_path.read_text(encoding='utf-8')

print(f'Loaded {len(texts)} judgment texts')

## 2. Run Outcome Parser

In [ ]:
parser = OutcomeParser()
results = parser.batch_parse(texts)

# Build results DataFrame
rows = []
for case_id, result in results.items():
    rows.append({
        'case_id': case_id,
        'outcome_label': result.label.value,
        'confidence': result.confidence,
        'needs_review': result.needs_review,
        'matched_text': result.matched_text,
        'num_matches': len(result.all_matches),
        'outcome_binary': parser.to_binary(result.label),
    })

outcomes_df = pd.DataFrame(rows)
print(outcomes_df['outcome_label'].value_counts())
print(f'\nCases needing review: {outcomes_df["needs_review"].sum()}')
print(f'Cases with binary label: {outcomes_df["outcome_binary"].notna().sum()}')

## 3. Confidence Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

outcomes_df['confidence'].hist(bins=20, ax=axes[0])
axes[0].set_title('Confidence Score Distribution')
axes[0].axvline(x=0.7, color='red', linestyle='--', label='Review threshold')
axes[0].legend()

outcomes_df['outcome_label'].value_counts().plot(kind='barh', ax=axes[1])
axes[1].set_title('Outcome Label Distribution')

plt.tight_layout()
plt.show()

## 4. Manual Review Queue

Export cases needing manual review for domain expert validation.
The Kenyan lawyer should review these and correct labels as needed.

In [ ]:
review_queue = outcomes_df[outcomes_df['needs_review']].copy()
review_queue['manual_label'] = ''  # To be filled by reviewer
review_queue['reviewer_notes'] = ''

review_path = settings.data.reference_dir / 'manual_review_queue.csv'
review_path.parent.mkdir(parents=True, exist_ok=True)
review_queue.to_csv(review_path, index=False)
print(f'Exported {len(review_queue)} cases for review to {review_path}')

## 5. Inter-Annotator Agreement

After two reviewers independently label 50 cases, compute Cohen's kappa.

In [ ]:
from sklearn.metrics import cohen_kappa_score

# TODO: Load reviewer annotations after manual labeling is complete
# reviewer_1 = pd.read_csv('data/reference/reviewer_1_labels.csv')
# reviewer_2 = pd.read_csv('data/reference/reviewer_2_labels.csv')
# kappa = cohen_kappa_score(reviewer_1['label'], reviewer_2['label'])
# print(f'Cohen\'s kappa: {kappa:.3f}')
# print(f'Target: >= 0.80')
# print(f'Result: {"PASS" if kappa >= 0.80 else "FAIL - review disagreements"}')

print('Waiting for manual review completion...')

## 6. Final Dataset Statistics

In [ ]:
binary_labeled = outcomes_df[outcomes_df['outcome_binary'].notna()]
print(f'Total cases with binary labels: {len(binary_labeled)}')
print(f'Positive (plaintiff wins): {(binary_labeled["outcome_binary"] == 1).sum()}')
print(f'Negative (defendant wins): {(binary_labeled["outcome_binary"] == 0).sum()}')
print(f'Base rate: {binary_labeled["outcome_binary"].mean():.3f}')
print(f'\nGate check: {"PASS" if len(binary_labeled) >= 400 else "FAIL"} (need >= 400, have {len(binary_labeled)})')